# SRA Toolkit

This notebook demonstrates the basics of installing and using the [SRA Toolkit](https://github.com/ncbi/sra-tools).

With the instructions in this notebook, you will be able to install the SRA Toolkit, configure access to SRA data, test the installation, inspect an accession, download it with `prefetch`, and convert it to FASTQ with `fasterq-dump`.

Commands that begin with `!` are shell commands run from inside the notebook, so if you want to run a command in your terminal, you may need to remove the `!`. File paths and some output text will vary depending on your computer and whether an accession has already been downloaded.

For more detailed reference material, see the [NCBI SRA Tutorials GitHub page](https://github.com/ncbi/sra-tutorials).

---

## 1. Choose demo accessions

SRA data are identified by accessions (with IDs like `SRR390728` or `SRR1553607`). This notebook stores the accessions in variables so the same commands can be reused with a different run later.

- `TESTING_RUN` is a very small run used only to confirm that the toolkit can retrieve and print reads.
- `RUN` is the example accession used for the inspect, download, and FASTQ conversion steps.

To adapt this notebook, change `RUN` to the accession you want to work with.

For this Jupyter Notebook, just select the code block below and type Shift+Enter or select **Run > Run Selected Cell** from the Notebook menu bar. You should see the **TASK COMPLETE** marker if the code block has successfully run.

In [ ]:
# Accessions used in this demo.
TESTING_RUN = "SRR390728"  # condensed run used to verify installation
RUN = "SRR1553607"         # insert your own favorite accession here


# Let you know that your accessions have been set.
print(f"{TESTING_RUN} is the current TESTING_RUN.")
print(f"{RUN} is the current RUN.")
print("\n\n-------------\nTASK COMPLETE\n-------------")

## 2. Install or confirm SRA Toolkit

Run **one** installation option only.

### *macOS/Linus Installation*

In hosted Linux notebooks, the direct NCBI download option is usually the simplest because it downloads a ready-to-use binary release and unpacks it into the current working directory.

The direct-download cell below retrieves the current Ubuntu build as a compressed `.tar.gz` file and extracts it. You may skip this cell if SRA Toolkit is already installed in your environment.

We will run this command for this demo notebook.

In [ ]:
# Linux/hosted Jupyter install: download and unpack the current Ubuntu build.
# Skip this cell if SRA Toolkit is already installed.

!wget -O sratoolkit.tar.gz https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz
!tar -xzf sratoolkit.tar.gz


# Add the unpacked SRA Toolkit bin folder to PATH for this notebook session.

import glob
import os

bin_dirs = glob.glob(os.path.abspath("sratoolkit.*/bin"))
if bin_dirs:
    os.environ["PATH"] = bin_dirs[0] + os.pathsep + os.environ["PATH"]
    print("Using SRA Toolkit bin:", bin_dirs[0])
else:
    print("No local sratoolkit bin found; assuming tools are already on PATH.")


# Let you know this task has completed.

print("\n\n-------------\nTASK COMPLETE\n-------------")

You'll know that SRA Toolkit has installed successfully if you are given an output saying `‘sratoolkit.tar.gz’ saved [89143120/89143120]`

After the archive is extracted, the command-line programs live inside a versioned `bin` folder (for example `sratoolkit.3.4.1-ubuntu64/bin`). This code block includes a command that adds a `bin` folder to the notebook session's `PATH`. This makes commands such as `prefetch`, `vdb-dump`, and `fasterq-dump` available from later notebook cells.

**NOTE:** This change applies to the current notebook session. If the runtime restarts, run this cell again.

### *macOS/Linus Installation (using Homebrew)*

On a local macOS or Linux machine, another easy way to install the SRA Toolkit is with [Homebrew](https://brew.sh/).

Once you have Homebrew installed installed (by pasting the installation snippet into your Terminal prompt), you can simply type `brew install sratoolkit` into your shell prompt to install the SRA Toolkit.

Just keep in mind that Homebrew is externally managed, and may not include the most updated version of the SRA Toolkit.

### *Windows Installation*

On Windows, download and unzip `sratoolkit.current-win64.zip`. Then either add the extracted `bin` folder to your system `PATH`, or open a command prompt from that `bin` folder and run the remaining SRA Toolkit commands there.



## 3. Confirm that your tools are available

The next cell checks that the key SRA Toolkit programs are available. Each line should print a file path. If a tool prints `not found`, the toolkit is not installed or its `bin` directory is not on `PATH`.

The commands checked here are:

- `fastq-dump`: older FASTQ export tool, useful here for a tiny installation test.
- `fasterq-dump`: current FASTQ conversion tool used for the main conversion step.
- `prefetch`: downloads SRA data into the local repository/cache.
- `vdb-dump`: inspects accession metadata and contents.
- `vdb-config`: views or edits SRA Toolkit configuration.

There is also a line that prints the installed version. Version information is useful when troubleshooting because command options and behavior can differ across toolkit releases.


In [ ]:
# Confirm the key SRA Toolkit commands are available.

import shutil

tools = [
    "fastq-dump",
    "fasterq-dump",
    "prefetch",
    "vdb-dump",
    "vdb-config",
]

for tool in tools:
    print(f"{tool}: {shutil.which(tool) or 'not found'}")

# Show the installed fasterq-dump version

!fasterq-dump --version

# Want to know what any specific tool does?
# Use the `--help` option for any tool to print out the manual.

#!fasterq-dump --help


## 4. Configure SRA Toolkit

SRA Toolkit uses configuration settings to control where data are cached and whether remote access is enabled. The interactive configuration program is `vdb-config -i`.

**Note:** Interactive terminal menus do not display inside Jupyter Notebooks, so these instructions are for when you want to run these commands on your local machine or server.

In the interactive menu, the important settings for this demo are:

- Enable **Remote Access** so the toolkit can retrieve accessions from NCBI/SRA storage.
- Enable **Local File Caching** so downloaded data can be reused instead of fetched again.
- Choose a user repository/cache location. For a new setup, choose an empty directory.
- Save and exit.

As a bonus, the `!vdb-config | head -60` command below prints the current configuration in XML format and shows only the first 60 lines. This is mainly to check that `vdb-config` runs successfully; you do not need to read every line when you are running this for yourself.

In [ ]:
# Interactive configuration; run in a terminal if it does not display well in Jupyter.
# Enable Remote Access, enable Local File Caching, choose an empty repository, save, and exit.

# !vdb-config -i

# Show current configuration after setup.

!vdb-config

## 5. Retrieve a test FASTQ record

This step confirms that the toolkit can retrieve reads and print FASTQ output. The command uses `fastq-dump` with two small options:

- `--stdout` sends the FASTQ records to the notebook output instead of writing files.
- `-X 2` limits the result to the first two spots so the test runs quickly.

If you see two FASTQ records printed, the toolkit is installed, accessible on `PATH`, and able to retrieve the test accession.

In [ ]:
# Print the first 2 FASTQ reads from a small accession
# Use your own accession (for example `fastq-dump --stdout -X 2 SRR390728`)
# In this example, we will be using the accession listed in the {TESTING_RUN} variable

!fastq-dump --stdout -X 2 {TESTING_RUN}

## 6. Inspect, download, and convert an accession

A common SRA Toolkit workflow is:

6.1. Inspect the accession with `vdb-dump --info`.
6.2. Download/cache the run with `prefetch`.
6.3. Check available disk space
6.4. Convert the downloaded run to FASTQ with `fasterq-dump`.
6.5. Confirm that the expected output files were created.

The example accession in this notebook is small enough for a quick demo, but the same workflow applies to larger runs.

### *6.1. Inspect accession metadata*

`vdb-dump` reports useful metadata before downloading or converting a run. The output can include the accession, local or remote path, file size, platform, number of spots/sequences, schema, and original format information.

The size information is especially useful because it helps you estimate whether you have enough disk space for the download and conversion.



In [ ]:
# Show accession metadata, including size information

!vdb-dump {RUN} --info


### *6.2. Download/cache the accession*

`prefetch` downloads the accession into the configured SRA repository/cache. This usually creates a local `.sra` or `.sralite` file that other SRA Toolkit commands can reuse.

If the run has already been downloaded, `prefetch` may report that the accession is "found locally." That is not an error; it means the cached copy is being reused.


In [ ]:
# Download the run into the configured SRA repository/cache

!prefetch {RUN}


### *6.3. Check available disk space*

FASTQ conversion can require much more temporary disk space than the size of the downloaded SRA file. A practical planning estimate from the original demo is to keep roughly **18x the accession size** free during `fasterq-dump` conversion. This accounts for temporary files plus the final FASTQ outputs.

`df -h .` reports human-readable free space for the filesystem that contains the current directory.


In [ ]:
# Check free disk space before conversion
# Keep roughly 18x accession size free for temp + FASTQ outputs

!df -h .


### *6.4. Convert to FASTQ*

`fasterq-dump` converts SRA data into FASTQ files, which are commonly used as input for downstream sequence analysis tools.

The `--split-files` option writes paired-end reads into separate files. For this example, the expected outputs are `SRR1553607_1.fastq` and `SRR1553607_2.fastq`. For single-end runs, output file names may differ.


In [ ]:
# Convert to FASTQ files in the current directory
# --split-files writes paired reads as RUN_1.fastq and RUN_2.fastq when paired-end data are present

!fasterq-dump {RUN} --split-files


### *6.5. Confirm the FASTQ outputs*

The `ls -lh` command lists the generated FASTQ files and their sizes in a human-readable format. Seeing the expected `.fastq` files confirms that conversion completed successfully.


In [ ]:
# Confirm FASTQ outputs were created

!ls -lh {RUN}*.fastq


## 7. Get help

Most SRA Toolkit tools include built-in help. Use `--help` or `-h` to see available options for a specific command.

For example, `fasterq-dump --help` shows options for choosing an output directory, changing the temporary directory, setting the number of threads, splitting reads, filtering reads, and writing FASTA instead of FASTQ.


In [ ]:
# Show the first part of the fasterq-dump help text

!fasterq-dump --help


---

If you have any more specific questions, please visit the [SRA Toolkit GitHub repository](https://github.com/ncbi/sra-tools) or email your questions to [sra@ncbi.nlm.nih.gov](mailto:sra@ncbi.nlm.nih.gov).